# AndroidVM — Google Colab (Android 4.4 x86)
This notebook boots the project's Android 4.4/API 19 **x86 Google APIs** emulator image and exposes its desktop through Selkies + a temporary Cloudflare Tunnel.

**First milestone:** verify that KitKat boots and that the browser stream works. The Colab runtime is temporary; this first version keeps the VM disk in `/content` and does not yet persist Android data to Drive.

Keep the tunnel URL private: anyone who has it can control the VM. Do not sign into sensitive accounts. Colab may disconnect or stop the runtime at any time.

In [ ]:
import os, subprocess, textwrap, pathlib, time, json, re, shutil

REPO_URL = "https://github.com/GoingTo2014/androidvm.git"
REPO_DIR = "/content/AndroidVM"
ANDROID_HOME = "/content/android-sdk"
AVD_HOME = "/content/android-avd"
VM_RAM_MB = 2048
VM_CPU_CORES = 2
DISPLAY_SIZE = "1280x800x24"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.environ.update({"ANDROID_HOME": ANDROID_HOME, "ANDROID_SDK_ROOT": ANDROID_HOME,
                   "ANDROID_AVD_HOME": AVD_HOME, "PATH": f"{ANDROID_HOME}/platform-tools:{ANDROID_HOME}/emulator:{ANDROID_HOME}/cmdline-tools/latest/bin:" + os.environ["PATH"]})
os.makedirs(AVD_HOME, exist_ok=True)
print("Repository:", REPO_DIR)
print("Android SDK:", ANDROID_HOME)


## 1. Install host dependencies
This uses Colab's Linux runtime. The L4 GPU is not required for this milestone: Android's display renderer will use SwiftShader.

In [ ]:
%%bash
set -euo pipefail
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
  openjdk-17-jdk xvfb pulseaudio pulseaudio-utils openbox xdotool wmctrl \
  curl unzip ca-certificates file libgl1 libnss3 libpulse0 \
  libxkbcommon-x11-0 libxcb-cursor0 libx11-xcb1 libxcb1
java -version


## 2. Download Android SDK tools and the KitKat x86 image
The selected image matches the existing GitHub Actions setup: `system-images;android-19;google_apis;x86`. Google may retire legacy SDK downloads; if installation fails, the SDK manager output will show why.

In [ ]:
%%bash
set -euo pipefail
export ANDROID_HOME=/content/android-sdk
export ANDROID_SDK_ROOT="$ANDROID_HOME"
mkdir -p "$ANDROID_HOME/cmdline-tools" /content/android-tools
if [ ! -x "$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager" ]; then
  curl -fL --retry 3 https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -o /content/android-tools/commandline-tools.zip
  rm -rf /content/android-tools/unpack "$ANDROID_HOME/cmdline-tools/latest"
  mkdir -p /content/android-tools/unpack
  unzip -q /content/android-tools/commandline-tools.zip -d /content/android-tools/unpack
  mv /content/android-tools/unpack/cmdline-tools "$ANDROID_HOME/cmdline-tools/latest"
fi
export PATH="$ANDROID_HOME/cmdline-tools/latest/bin:$PATH"
yes | sdkmanager --sdk_root="$ANDROID_HOME" --licenses >/dev/null || true
sdkmanager --sdk_root="$ANDROID_HOME" --install "platform-tools" "emulator" "system-images;android-19;google_apis;x86"
test -f "$ANDROID_HOME/system-images/android-19/google_apis/x86/system.img"
"$ANDROID_HOME/emulator/emulator" -version


## 3. Create the Android 4.4 AVD and check acceleration
The notebook automatically uses KVM only if `/dev/kvm` is available and the emulator's acceleration check succeeds. Otherwise it falls back to software CPU emulation, which can be very slow.

In [ ]:
%%bash
set -euo pipefail
export ANDROID_HOME=/content/android-sdk ANDROID_SDK_ROOT=/content/android-sdk ANDROID_AVD_HOME=/content/android-avd
export PATH="$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools:$ANDROID_HOME/emulator:$PATH"
mkdir -p "$ANDROID_AVD_HOME"
if [ ! -f "$ANDROID_AVD_HOME/KitKat.ini" ]; then
  echo no | avdmanager create avd --name KitKat --package "system-images;android-19;google_apis;x86" --force
fi
cat >> "$ANDROID_AVD_HOME/KitKat.avd/config.ini" <<'EOF'
hw.cpu.ncore=2
hw.ramSize=2048
hw.keyboard=yes
hw.gpu.enabled=yes
hw.gpu.mode=swiftshader
showDeviceFrame=no
skin.dynamic=no
disk.dataPartition.size=2G
EOF
echo "--- KVM device ---"
ls -l /dev/kvm 2>/dev/null || echo "/dev/kvm is not exposed"
if [ -e /dev/kvm ] && "$ANDROID_HOME/emulator/emulator" -accel-check; then echo on > /content/androidvm-accel-mode; else echo off > /content/androidvm-accel-mode; fi
echo "Selected acceleration: $(cat /content/androidvm-accel-mode)"


## 4. Start the virtual display, audio and Android
Run this cell once. If you rerun it, stop the previous emulator process first or restart the Colab runtime.

In [ ]:
%%bash
set -euo pipefail
export ANDROID_HOME=/content/android-sdk ANDROID_SDK_ROOT=/content/android-sdk ANDROID_AVD_HOME=/content/android-avd
export PATH="$ANDROID_HOME/platform-tools:$ANDROID_HOME/emulator:$PATH"
mkdir -p /content/androidvm-logs
if ! pgrep -x Xvfb >/dev/null; then Xvfb :99 -screen 0 1280x800x24 -ac +extension GLX +extension RANDR +extension RENDER +extension XFIXES +extension XTEST -nolisten tcp > /content/androidvm-logs/xvfb.log 2>&1 & fi
export DISPLAY=:99
sleep 2
openbox --sm-disable > /content/androidvm-logs/openbox.log 2>&1 &
pulseaudio --start --exit-idle-time=-1 || true
pactl load-module module-null-sink sink_name=vm_audio sink_properties=device.description=AndroidVM >/dev/null 2>&1 || true
pactl set-default-sink vm_audio >/dev/null 2>&1 || true
ACCEL=$(cat /content/androidvm-accel-mode)
if [ "$ACCEL" = on ]; then ACCEL_ARGS=(-accel on); else ACCEL_ARGS=(-accel off); fi
nohup env DISPLAY=:99 PULSE_SINK=vm_audio QEMU_AUDIO_DRV=pa "$ANDROID_HOME/emulator/emulator" \
  -avd KitKat -port 5554 "${ACCEL_ARGS[@]}" -no-snapshot -no-boot-anim \
  -gpu swiftshader -memory 2048 -cores 2 -writable-system \
  > /content/androidvm-logs/emulator.log 2>&1 < /dev/null &
echo "Emulator started. Waiting for Android boot..."
adb start-server
for i in $(seq 1 240); do
  if adb devices | grep -q 'emulator-5554[[:space:]]*device' && [ "$(adb -s emulator-5554 shell getprop sys.boot_completed 2>/dev/null | tr -d '\r')" = 1 ]; then break; fi
  if ! pgrep -f 'qemu-system|/emulator.*-avd KitKat' >/dev/null; then tail -n 100 /content/androidvm-logs/emulator.log; exit 1; fi
  sleep 2
done
test "$(adb -s emulator-5554 shell getprop sys.boot_completed | tr -d '\r')" = 1 || { echo 'Android did not finish booting'; tail -n 120 /content/androidvm-logs/emulator.log; exit 1; }
adb -s emulator-5554 shell settings put system screen_off_timeout 2147483647 || true
adb -s emulator-5554 shell input keyevent 82 || true
echo "Android ABI: $(adb -s emulator-5554 shell getprop ro.product.cpu.abi | tr -d '\r')"
adb -s emulator-5554 shell getprop ro.build.version.release


## 5. Install APKs from the repository folders (optional)
Regular APKs in `apks/` are installed as user apps. System APK injection is intentionally deferred until the first boot/stream test passes.

In [ ]:
%%bash
set -euo pipefail
shopt -s nullglob
for apk in /content/AndroidVM/apks/*.apk; do echo "Installing $(basename "$apk")"; adb -s emulator-5554 install -r "$apk" || echo "WARNING: install failed: $apk"; done
echo "APK installation pass complete."


## 6. Start Selkies and the temporary Cloudflare URL
The URL is temporary and grants control of the Android desktop. Keep it private. Colab's session limits still apply.

In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/androidvm-logs
SELKIES_APP=/content/selkies-2.0.0-x86_64.AppImage
if [ ! -x "$SELKIES_APP" ]; then
  curl -fL --retry 3 "https://github.com/selkies-project/selkies/releases/download/2.0.0/selkies-2.0.0-x86_64.AppImage" -o "$SELKIES_APP"
  echo "8dcdbe239dc8930545819aae510c45283cf25922250b6b7348ff2fd790c41ed8  $SELKIES_APP" | sha256sum --check
  chmod +x "$SELKIES_APP"
fi
"$SELKIES_APP" --appimage-extract-and-run --version
nohup env DISPLAY=:99 "$SELKIES_APP" --appimage-extract-and-run \
  --addr=127.0.0.1 --port=6080 --enable-https=false --enable-basic-auth=false \
  --mode=websockets --enable-dual-mode=true --encoder=h264enc --use-cpu=true \
  --enable-resize=false --framerate=30,8-30 \
  --audio-enabled=true --audio-on-start=true --audio-device-name=vm_audio.monitor \
  > /content/androidvm-logs/selkies.log 2>&1 < /dev/null &
for i in $(seq 1 60); do if curl -fsS http://127.0.0.1:6080/ | grep -qi Selkies; then break; fi; sleep 2; done
curl -fsS http://127.0.0.1:6080/ | grep -qi Selkies || { tail -n 100 /content/androidvm-logs/selkies.log; exit 1; }
if ! command -v cloudflared >/dev/null 2>&1; then
  curl -fL --retry 3 "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb" -o /content/cloudflared.deb
  dpkg -i /content/cloudflared.deb
fi
nohup cloudflared tunnel --no-autoupdate --url http://127.0.0.1:6080 > /content/androidvm-logs/cloudflared.log 2>&1 < /dev/null &
for i in $(seq 1 60); do URL=$(grep -Eo 'https://[-a-z0-9]+\.trycloudflare\.com' /content/androidvm-logs/cloudflared.log | head -n 1 || true); [ -n "$URL" ] && break; sleep 2; done
if [ -z "${URL:-}" ]; then tail -n 100 /content/androidvm-logs/cloudflared.log; exit 1; fi
echo "ANDROIDVM COLAB IS READY: $URL"
echo "Keep this URL private; anyone with it can control the VM."


## Troubleshooting / stop
- If the emulator fails to start, inspect `/content/androidvm-logs/emulator.log`.
- If the page does not load, inspect `selkies.log` and `cloudflared.log` in `/content/androidvm-logs/`.
- If `/dev/kvm` is missing, the VM uses software CPU emulation and may be very slow.
- Restarting or disconnecting the Colab runtime terminates the VM and tunnel. Android changes are not persistent in this first milestone.

To stop the processes manually, run: `!pkill -f cloudflared; !pkill -f selkies; !pkill -f emulator` (or restart the runtime).